In [4]:
import os, json, joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# ============================================================
# 0) CONFIG
# ============================================================
model_dir = "./Ridge_OTU_Trained_Models/"   # contains ridge_otu_models.pkl + ridge_training_metadata.json
ukb_metadata_path = "variable_mapping/ukb_as_agp_metadata.csv"
ukb_index_col = "sample_name"

# outcome
cognition_col = "fluid_intelligence_score"   # change if needed

out_dir = "./hongrui_result/Ridge_CV_Results/"
os.makedirs(out_dir, exist_ok=True)

# ============================================================
# 1) LOAD SAVED RIDGE OTU MODELS + TRAINING METADATA
# ============================================================
ridge_models = joblib.load(os.path.join(model_dir, "ridge_otu_models.pkl"))
with open(os.path.join(model_dir, "ridge_training_metadata.json"), "r") as f:
    train_meta = json.load(f)

train_cols = train_meta["train_cols"]          # numeric metadata used in training X
saved_otus = train_meta["saved_otus"]          # OTU targets saved (should be ~34)
print(f"Loaded {len(ridge_models)} ridge OTU models.")
print(f"Training expects {len(train_cols)} numeric metadata columns.")
print(f"Will predict {len(saved_otus)} OTUs.")

# ============================================================
# 2) LOAD UKB METADATA
# ============================================================
ukb_df = pd.read_csv(ukb_metadata_path, index_col=ukb_index_col)
ukb_df["age_corrected"] = pd.to_numeric(ukb_df["age_corrected"], errors="coerce")

if cognition_col not in ukb_df.columns:
    raise ValueError(f"cognition_col='{cognition_col}' not found in ukb_df.")

# ============================================================
# 2.5) ROW FILTERING BY MISSINGNESS (same style as projection script)
# ============================================================
skip_cols = [
    "sugar_sweetened_drink_frequency",
    "free_sugar_scaled_0_5",
    "artificial_sweeteners",
    "one_liter_of_water_a_day_frequency",
    "olive_oil",
    "prepared_meals_frequency",
    "ready_to_eat_meals_frequency",
    "probiotic_frequency",
    "whole_eggs",
    "vitamin_b_supplement_frequency",
    "vitamin_d_supplement_frequency",
    "sugary_sweets_frequency"
]
max_missing_frac = 0.10

# covariates (EXACT pattern: numeric + categorical)
cov_num = ["age_corrected"]     # continuous
cov_cat = ["race"]              # categorical (dummies)

required_for_regression = [cognition_col] + cov_num

def filter_rows_by_missingness(df, *, skip_cols=(), required_cols=(), max_missing_frac=0.10):
    skip_cols = list(skip_cols)
    required_cols = list(required_cols)

    missing_skip = [c for c in skip_cols if c not in df.columns]
    if missing_skip:
        print(f"[WARN] skip_cols not in dataframe (ignored): {missing_skip[:10]}{'...' if len(missing_skip)>10 else ''}")

    cols_check = [c for c in df.columns if c not in set(skip_cols)]
    if len(cols_check) == 0:
        raise ValueError("After applying skip_cols, cols_check is empty. Reduce skip_cols.")

    missing_frac = df[cols_check].isna().mean(axis=1)
    n_in = df.shape[0]
    mask = missing_frac <= max_missing_frac
    df_f = df.loc[mask].copy()

    required_present = [c for c in required_cols if c in df_f.columns]
    df_f = df_f.dropna(subset=required_present)

    report = {
        "n_in": int(n_in),
        "n_out": int(df_f.shape[0]),
        "dropped": int(n_in - df_f.shape[0]),
        "cols_check_n": int(len(cols_check)),
        "missing_frac_summary": missing_frac.describe().to_dict()
    }
    return df_f, report

ukb_df, miss_report = filter_rows_by_missingness(
    ukb_df,
    skip_cols=skip_cols,
    required_cols=required_for_regression,
    max_missing_frac=max_missing_frac
)

print("\n[ROW FILTER REPORT]")
print(f"Input rows:   {miss_report['n_in']}")
print(f"Output rows:  {miss_report['n_out']}")
print(f"Dropped rows: {miss_report['dropped']}")
print(f"Cols counted toward missingness: {miss_report['cols_check_n']}")

# ============================================================
# 3) BUILD UKB X MATRIX WITH EXACT TRAINING COLUMNS
# ============================================================
X_ukb = ukb_df.reindex(columns=train_cols)
X_ukb = X_ukb.apply(pd.to_numeric, errors="coerce")

missing_cols = [c for c in train_cols if c not in ukb_df.columns]
print(f"UKB is missing {len(missing_cols)} / {len(train_cols)} training columns.")
if missing_cols:
    print("First 25 missing training columns:", missing_cols[:25])

missing_rate = float(np.mean(pd.isna(X_ukb.values)))
print(f"Overall missing rate in X_ukb: {missing_rate*100:.2f}%")

# ============================================================
# 4) PREDICT OTU_hat FOR UKB
# ============================================================
otu_hat = pd.DataFrame(index=ukb_df.index)

for otu in saved_otus:
    if otu not in ridge_models:
        raise KeyError(f"OTU '{otu}' is in saved_otus but missing from ridge_otu_models.pkl.")
    otu_hat[f"{otu}__hat"] = ridge_models[otu].predict(X_ukb)

otu_hat_path = os.path.join(out_dir, "ukb_predicted_microbiome_otus.csv")
otu_hat.to_csv(otu_hat_path)
print(f"Saved predicted OTUs to: {otu_hat_path}")

# ============================================================
# 5) BUILD COVARIATES (numeric + categorical dummies)
# ============================================================
cov_df_num = ukb_df[cov_num].apply(pd.to_numeric, errors="coerce")
cov_df_cat = pd.get_dummies(
    ukb_df[cov_cat].astype("string").fillna("MISSING"),
    prefix=cov_cat,
    drop_first=True,
    dtype=float
)
cov_df = pd.concat([cov_df_num, cov_df_cat], axis=1)
cov_cols = list(cov_df.columns)

# ============================================================
# 6) COMBINED OLS (HC3): cognition ~ ALL OTU_hat + covariates
# ============================================================
y = pd.to_numeric(ukb_df[cognition_col], errors="coerce")

otu_cols = [c for c in otu_hat.columns if c.endswith("__hat")]
otu_hat_num = otu_hat[otu_cols].apply(pd.to_numeric, errors="coerce")

X_full = pd.concat([otu_hat_num, cov_df], axis=1)
X_full = sm.add_constant(X_full, has_constant="add")
X_full = X_full.replace([np.inf, -np.inf], np.nan).apply(pd.to_numeric, errors="coerce")

data_full = pd.concat([y.rename("y"), X_full], axis=1).dropna()
print("Rows used in FULL regression:", data_full.shape[0], "Cols:", data_full.shape[1])

y_clean = data_full["y"].astype(float)
X_clean = data_full.drop(columns=["y"]).astype(float)

fit_full = sm.OLS(y_clean, X_clean).fit(cov_type="HC3")
print(fit_full.summary())

# ============================================================
# 7) BASELINE OLS (HC3): cognition ~ covariates only
# ============================================================
X_base = sm.add_constant(cov_df, has_constant="add")
X_base = X_base.replace([np.inf, -np.inf], np.nan).apply(pd.to_numeric, errors="coerce")

data_base = pd.concat([y.rename("y"), X_base], axis=1).dropna()
print("Rows used in BASE regression:", data_base.shape[0], "Cols:", data_base.shape[1])

fit_base = sm.OLS(
    data_base["y"].astype(float),
    data_base.drop(columns=["y"]).astype(float)
).fit(cov_type="HC3")

# If samples differ (they often will), restrict to common index for fair ΔR²
common_idx = data_full.index.intersection(data_base.index)
fit_full_common = sm.OLS(
    data_full.loc[common_idx, "y"].astype(float),
    data_full.loc[common_idx].drop(columns=["y"]).astype(float)
).fit(cov_type="HC3")

fit_base_common = sm.OLS(
    data_base.loc[common_idx, "y"].astype(float),
    data_base.loc[common_idx].drop(columns=["y"]).astype(float)
).fit(cov_type="HC3")

delta_r2 = float(fit_full_common.rsquared - fit_base_common.rsquared)
print("\n=== Model comparison (same sample) ===")
print("R2 base:", float(fit_base_common.rsquared))
print("R2 full:", float(fit_full_common.rsquared))
print("ΔR2 (OTUs add):", delta_r2)

# ============================================================
# 8) JOINT WALD TEST: all OTU_hat coefficients = 0
# ============================================================
param_names = list(fit_full.params.index)
otu_in_model = [c for c in otu_cols if c in param_names]

R = np.zeros((len(otu_in_model), len(param_names)))
for i, col in enumerate(otu_in_model):
    R[i, param_names.index(col)] = 1.0

wald = fit_full.wald_test(R, scalar=True)
print("\n=== Joint test (all OTUs) ===")
print("Wald stat:", float(wald.statistic))
print("p-value:", float(wald.pvalue))

# ============================================================
# 9) SAVE COEFFICIENT TABLE (OTUs only) + BH FDR across OTUs
# ============================================================
coef_df = pd.DataFrame({
    "otu_hat": otu_in_model,
    "beta": [float(fit_full.params[c]) for c in otu_in_model],
    "se_hc3": [float(fit_full.bse[c]) for c in otu_in_model],
    "t_hc3": [float(fit_full.tvalues[c]) for c in otu_in_model],
    "p_value": [float(fit_full.pvalues[c]) for c in otu_in_model],
})

rej, qvals, _, _ = multipletests(coef_df["p_value"].values, alpha=0.05, method="fdr_bh")
coef_df["q_value_BH"] = qvals
coef_df["reject_FDR_0p05"] = rej

coef_df = coef_df.sort_values("p_value")
coef_path = os.path.join(out_dir, "combined_model_otu_coefficients_hc3.csv")
coef_df.to_csv(coef_path, index=False)
print(f"Saved OTU coefficient table to: {coef_path}")

# Save summaries
with open(os.path.join(out_dir, "combined_model_full_summary.txt"), "w") as f:
    f.write(fit_full.summary().as_text())
with open(os.path.join(out_dir, "combined_model_base_summary.txt"), "w") as f:
    f.write(fit_base.summary().as_text())

with open(os.path.join(out_dir, "combined_model_metrics.txt"), "w") as f:
    f.write(f"R2_base_common\t{float(fit_base_common.rsquared)}\n")
    f.write(f"R2_full_common\t{float(fit_full_common.rsquared)}\n")
    f.write(f"delta_R2\t{delta_r2}\n")
    f.write(f"wald_stat_all_otus\t{float(wald.statistic)}\n")
    f.write(f"wald_p_all_otus\t{float(wald.pvalue)}\n")

print("Done.")

Loaded 32 ridge OTU models.
Training expects 29 numeric metadata columns.
Will predict 32 OTUs.
[WARN] skip_cols not in dataframe (ignored): ['sugary_sweets_frequency']

[ROW FILTER REPORT]
Input rows:   502244
Output rows:  160568
Dropped rows: 341676
Cols counted toward missingness: 37
UKB is missing 0 / 29 training columns.
Overall missing rate in X_ukb: 18.51%
Saved predicted OTUs to: ./hongrui_result/Ridge_CV_Results/ukb_predicted_microbiome_otus.csv
Rows used in FULL regression: 160568 Cols: 38


/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions/my_sklearn_env/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 36, but rank is 34
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.136
Model:                            OLS   Adj. R-squared:                  0.136
Method:                 Least Squares   F-statistic:                 4.358e+04
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        12:04:07   Log-Likelihood:            -3.3862e+05
No. Observations:              160568   AIC:                         6.773e+05
Df Residuals:                  160534   BIC:                         6.776e+05
Df Model:                          33                                         
Covariance Type:                  HC3                                         
                                                                       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions/my_sklearn_env/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 32, but rank is 30
  warnings.warn('covariance of constraints does not have full '


In [7]:
# ============================================================
# 10) VISUALIZATION: covariate-adjusted linear relationship
#     (microbiome proxy score vs fluid intelligence)
# ============================================================
import matplotlib
matplotlib.use("Agg")  # safe for HPC / headless runs
import matplotlib.pyplot as plt

# --- Use the fair, matched sample you already created ---
idx = common_idx  # computed earlier for fair comparisons

# Outcome on the common sample
y_common = data_full.loc[idx, "y"].astype(float)

# Covariate design matrix on the common sample (includes const)
X_base_common = data_base.loc[idx].drop(columns=["y"]).astype(float)

# Residualize outcome for covariates: y_resid = y - E[y|cov]
yhat_base = fit_base_common.predict(X_base_common)
y_resid = (y_common - yhat_base).to_numpy()

# Identify OTU columns in the *common* full model, then build a weighted "microbiome score"
param_names_common = list(fit_full_common.params.index)
otu_in_common = [c for c in otu_cols if c in param_names_common]  # otu_cols defined earlier

beta_otu = fit_full_common.params.loc[otu_in_common].astype(float)
otu_mat = data_full.loc[idx, otu_in_common].astype(float)

# Microbiome proxy score = sum_j beta_j * OTUhat_j
otu_score = (otu_mat.to_numpy() @ beta_otu.to_numpy())

# Residualize microbiome score for covariates: score_resid = score - E[score|cov]
score_hat = sm.OLS(otu_score, X_base_common).fit().predict(X_base_common)
otu_score_resid = (otu_score - score_hat).to_numpy()

# (Optional) Standardize so slope is in SD units
def zscore(v):
    v = np.asarray(v, dtype=float)
    return (v - np.nanmean(v)) / np.nanstd(v)

otu_score_resid_z = zscore(otu_score_resid)
y_resid_z = zscore(y_resid)

# Fit simple line for annotation
vis_fit = sm.OLS(y_resid_z, sm.add_constant(otu_score_resid_z)).fit()
slope = float(vis_fit.params[1])
pval = float(vis_fit.pvalues[1])
r2 = float(vis_fit.rsquared)
n = int(len(y_resid_z))

# Plot: use hexbin for big N, scatter for smaller N
plt.figure(figsize=(7, 6))
if n > 50000:
    plt.hexbin(otu_score_resid_z, y_resid_z, gridsize=60, mincnt=1)
else:
    plt.scatter(otu_score_resid_z, y_resid_z, alpha=0.25, s=10)

xs = np.linspace(np.nanmin(otu_score_resid_z), np.nanmax(otu_score_resid_z), 200)
plt.plot(xs, vis_fit.params[0] + vis_fit.params[1] * xs, linewidth=2)

plt.xlabel("Residualized microbiome proxy score (z)")
plt.ylabel(f"Residualized {cognition_col} (z)")
plt.title(
    f"Partial relationship (covariate-adjusted)\n"
    f"slope={slope:.4g}, p={pval:.2e}, R²={r2:.3g}, n={n}"
)
plt.tight_layout()

png_path = os.path.join(out_dir, "partial_relationship_microbiome_vs_fluid_intelligence.png")
pdf_path = os.path.join(out_dir, "partial_relationship_microbiome_vs_fluid_intelligence.pdf")
plt.savefig(png_path, dpi=300)
plt.savefig(pdf_path)

print(f"[PLOT SAVED] {png_path}")
print(f"[PLOT SAVED] {pdf_path}")
plt.close()

[PLOT SAVED] ./hongrui_result/Ridge_CV_Results/partial_relationship_microbiome_vs_fluid_intelligence.png
[PLOT SAVED] ./hongrui_result/Ridge_CV_Results/partial_relationship_microbiome_vs_fluid_intelligence.pdf


In [8]:
# 10) "Strength of Relationships" plot
#     Proportional Effect = normalized partial R² share
#     Optional bootstrap CIs
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------
# Settings
# ----------------------------
PLOT_WHAT = "all"        # "all" or "otus"
TOP_K = 25               # how many rows to show
N_BOOT = 200             # 0 = no CI; otherwise bootstrap replicates (100–300 is typical)
SEED = 42

DOT_COLOR = "#2a9d8f"    # teal
CI_COLOR  = "#e76f51"    # orange

plt.style.use("seaborn-v0_8-whitegrid")

# Helper: proportional effect from a fitted statsmodels result
def proportional_effect_from_fit(fit, preds):
    t = pd.Series(fit.tvalues, index=fit.params.index).reindex(preds)
    t = t.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    df = float(fit.df_resid)
    partial_r2 = (t ** 2) / (t ** 2 + df)

    total = float(partial_r2.sum())
    if total <= 0:
        prop = partial_r2 * 0.0
    else:
        prop = partial_r2 / total
    return prop

def pretty_name(v):
    if v.endswith("__hat"):
        return v.replace("__hat", "")
    if v.startswith("race_"):
        return "race: " + v.replace("race_", "")
    return v

# Choose predictors to plot
all_params = [p for p in fit_full_common.params.index if p != "const"]

if PLOT_WHAT == "otus":
    preds = [p for p in otu_in_model if p in all_params]
else:
    preds = all_params

# Point estimates (on full common sample)
prop0 = proportional_effect_from_fit(fit_full_common, preds)

# Select TOP_K
top_preds = list(prop0.sort_values(ascending=False).head(TOP_K).index)
prop0_top = prop0.reindex(top_preds)

# Bootstrap CIs (optional)
lo = prop0_top.copy()
hi = prop0_top.copy()

if N_BOOT and N_BOOT > 0:
    rng = np.random.default_rng(SEED)

    # Data on the common sample (already NA-clean)
    df_vis = data_full.loc[common_idx].copy()
    y_vis = df_vis["y"].astype(float)
    X_vis = df_vis.drop(columns=["y"]).astype(float)

    boot_mat = np.full((N_BOOT, len(top_preds)), np.nan, dtype=float)

    for b in range(N_BOOT):
        idx_pos = rng.integers(0, len(df_vis), size=len(df_vis))
        yb = y_vis.iloc[idx_pos]
        Xb = X_vis.iloc[idx_pos]

        fb = sm.OLS(yb, Xb).fit(cov_type="HC3")
        pb = proportional_effect_from_fit(fb, preds).reindex(top_preds).to_numpy()
        boot_mat[b, :] = pb

    lo[:] = np.nanpercentile(boot_mat, 2.5, axis=0)
    hi[:] = np.nanpercentile(boot_mat, 97.5, axis=0)

# Build plotting table
plot_df = pd.DataFrame({
    "var": top_preds,
    "label": [pretty_name(v) for v in top_preds],
    "effect": prop0_top.values,
    "lo": lo.values,
    "hi": hi.values,
}).sort_values("effect", ascending=True)  # smallest at bottom like your example

# ----------------------------
# Plot
# ----------------------------
fig_h = max(5, 0.45 * len(plot_df) + 1.5)
plt.figure(figsize=(10, fig_h))

ypos = np.arange(len(plot_df))
x = plot_df["effect"].to_numpy()
xerr_left = x - plot_df["lo"].to_numpy()
xerr_right = plot_df["hi"].to_numpy() - x

plt.axvline(0.0, linestyle="--", linewidth=2, color=CI_COLOR)

plt.errorbar(
    x, ypos,
    xerr=[xerr_left, xerr_right],
    fmt="o",
    markersize=10,
    markerfacecolor=DOT_COLOR,
    markeredgecolor=DOT_COLOR,
    ecolor=CI_COLOR,
    elinewidth=5,
    capsize=0
)

plt.yticks(ypos, plot_df["label"])
plt.xlabel("Proportional Effect")
plt.title("Strength of Relationships")
plt.tight_layout()

png_path = os.path.join(out_dir, f"strength_of_relationships_{PLOT_WHAT}_top{TOP_K}.png")
pdf_path = os.path.join(out_dir, f"strength_of_relationships_{PLOT_WHAT}_top{TOP_K}.pdf")
csv_path = os.path.join(out_dir, f"strength_of_relationships_{PLOT_WHAT}_top{TOP_K}.csv")

plt.savefig(png_path, dpi=300)
plt.savefig(pdf_path)
plt.close()

plot_df.to_csv(csv_path, index=False)

print(f"[SAVED] {png_path}")
print(f"[SAVED] {pdf_path}")
print(f"[SAVED] {csv_path}")

[SAVED] ./hongrui_result/Ridge_CV_Results/strength_of_relationships_all_top25.png
[SAVED] ./hongrui_result/Ridge_CV_Results/strength_of_relationships_all_top25.pdf
[SAVED] ./hongrui_result/Ridge_CV_Results/strength_of_relationships_all_top25.csv


In [9]:
# ============================================================
# Ridge coefficient variability plot (boxplot + dots)
#   - Fits Ridge on the association model: y ~ covariates + OTU_hat (combined)
#   - Uses bootstrap resampling to get coefficient variability
#   - Plots top-K features by |coef| on the full sample
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# ----------------------------
# Settings
# ----------------------------
PLOT_WHAT = "all"     # "all" or "otus"
TOP_K = 25
ALPHA = 1.0           # "small regularization" when X is standardized; try 0.1, 1, 10
N_BOOT = 200          # bootstrap replicates (e.g., 200-500). set 0 to skip variability
SEED = 42

BOX_COLOR = "#2a9d8f"
DOT_COLOR = "black"

# ----------------------------
# Build X, y on the matched sample you already use
# ----------------------------
df = data_full.loc[common_idx].copy()          # assumes you already created common_idx
y = df["y"].astype(float).to_numpy()
X_df = df.drop(columns=["y"]).astype(float)

# Identify OTU columns (fallback if you didn't keep otu_cols)
otu_cols_local = [c for c in X_df.columns if str(c).endswith("__hat")]

if PLOT_WHAT == "otus":
    feats = otu_cols_local
else:
    feats = list(X_df.columns)

X_df = X_df[feats]
feature_names = list(X_df.columns)

# ----------------------------
# Ridge pipeline (standardize X so ALPHA is meaningful)
# ----------------------------
ridge_pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("ridge", Ridge(alpha=ALPHA, fit_intercept=True, random_state=SEED))
])

# Fit once on full sample (for ranking + point estimate)
ridge_pipe.fit(X_df.to_numpy(), y)
coef_full = pd.Series(ridge_pipe.named_steps["ridge"].coef_, index=feature_names)

# Choose TOP_K by absolute coefficient on full sample
top_feats = list(coef_full.abs().sort_values(ascending=False).head(TOP_K).index)
coef_full_top = coef_full.reindex(top_feats)

# ----------------------------
# Bootstrap coefficient distributions
# ----------------------------
boot_coefs = None
if N_BOOT and N_BOOT > 0:
    rng = np.random.default_rng(SEED)
    X = X_df.to_numpy()
    n = X.shape[0]

    boot_coefs = np.zeros((N_BOOT, len(top_feats)), dtype=float)

    top_idx = [feature_names.index(f) for f in top_feats]

    for b in range(N_BOOT):
        samp = rng.integers(0, n, size=n)  # sample rows with replacement
        ridge_pipe.fit(X[samp, :], y[samp])
        boot_coefs[b, :] = ridge_pipe.named_steps["ridge"].coef_[top_idx]

# ----------------------------
# Pretty labels (optional)
# ----------------------------
def pretty_name(v):
    v = str(v)
    if v.endswith("__hat"):
        v = v.replace("__hat", "")
    if v.startswith("race_"):
        v = "race: " + v.replace("race_", "")
    return v

labels = [pretty_name(v) for v in top_feats]

# Sort for plotting (small -> large like your example)
order = np.argsort(coef_full_top.values)
top_feats_ord = [top_feats[i] for i in order]
labels_ord = [labels[i] for i in order]
coef_full_ord = coef_full_top.values[order]

if boot_coefs is not None:
    boot_ord = boot_coefs[:, order]

# ----------------------------
# Plot
# ----------------------------
plt.style.use("seaborn-v0_8-whitegrid")
fig_h = max(5, 0.45 * len(top_feats_ord) + 1.5)
fig, ax = plt.subplots(figsize=(10, fig_h))

ax.axvline(0.0, color="gray", linewidth=1.5)

ypos = np.arange(len(top_feats_ord))

if boot_coefs is not None:
    # Boxplots
    bp = ax.boxplot(
        [boot_ord[:, i] for i in range(boot_ord.shape[1])],
        vert=False,
        positions=ypos,
        widths=0.6,
        patch_artist=True,
        showfliers=False
    )
    for box in bp["boxes"]:
        box.set_facecolor(BOX_COLOR)
        box.set_alpha(0.75)
    for med in bp["medians"]:
        med.set_color("black")
        med.set_linewidth(2)

    # Overlay bootstrap points (jittered)
    for i in range(boot_ord.shape[1]):
        xs = boot_ord[:, i]
        jitter = (np.random.default_rng(SEED + i).random(xs.shape[0]) - 0.5) * 0.18
        ax.scatter(xs, ypos[i] + jitter, s=10, alpha=0.35, color=DOT_COLOR, linewidths=0)

# Overlay the full-sample coefficient as a prominent dot
ax.scatter(coef_full_ord, ypos, s=60, color="black", zorder=5)

ax.set_yticks(ypos)
ax.set_yticklabels(labels_ord)
ax.set_xlabel("Coefficient importance (standardized X)")
ax.set_title("Coefficient importance and its variability")
fig.suptitle("Ridge model, small regularization", y=0.99)

fig.tight_layout()

png_path = os.path.join(out_dir, f"ridge_coef_variability_{PLOT_WHAT}_top{TOP_K}_a{ALPHA}.png")
pdf_path = os.path.join(out_dir, f"ridge_coef_variability_{PLOT_WHAT}_top{TOP_K}_a{ALPHA}.pdf")
fig.savefig(png_path, dpi=300)
fig.savefig(pdf_path)
plt.close(fig)

print(f"[SAVED] {png_path}")
print(f"[SAVED] {pdf_path}")

[SAVED] ./hongrui_result/Ridge_CV_Results/ridge_coef_variability_all_top25_a1.0.png
[SAVED] ./hongrui_result/Ridge_CV_Results/ridge_coef_variability_all_top25_a1.0.pdf
